# QAI Control — Preparação Inicial dos Dados


In [ ]:
import zipfile
from pathlib import Path
import pandas as pd
import numpy as np


try:
    from google.colab import files

    print("Selecione o arquivo .zip da base:")
    uploaded = files.upload()

    zip_names = [name for name in uploaded.keys() if name.lower().endswith(".zip")]

    if not zip_names:
        raise FileNotFoundError("Nenhum arquivo .zip foi enviado.")

    ZIP_PATH = Path(zip_names[0])

except ImportError:
    # Caso o notebook seja executado fora do Colab
    candidatos = [
        Path("synthetic_ai_usage_dataset (1) (1).zip"),
        Path("/mnt/data/synthetic_ai_usage_dataset (1) (1).zip"),
    ]

    ZIP_PATH = next((p for p in candidatos if p.exists()), None)

    if ZIP_PATH is None:
        raise FileNotFoundError(
            "Arquivo .zip não encontrado. Coloque-o na mesma pasta do notebook."
        )

print(f"Arquivo utilizado: {ZIP_PATH}")

# Extrai a base
EXTRACT_DIR = Path("qai_dataset")
EXTRACT_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

# Localiza automaticamente a pasta que contém os CSVs
csv_files = list(EXTRACT_DIR.rglob("*.csv"))

if not csv_files:
    raise FileNotFoundError("Nenhum arquivo CSV foi encontrado dentro do .zip.")

CSV_DIR = csv_files[0].parent

# Carrega todas as tabelas CSV
dfs = {
    arquivo.name: pd.read_csv(arquivo, low_memory=False)
    for arquivo in sorted(CSV_DIR.glob("*.csv"))
}

print(f"\n{len(dfs)} tabelas carregadas com sucesso.")
print("Pasta dos CSVs:", CSV_DIR)

sorted(dfs.keys())


Selecione o arquivo .zip da base:


## 1. Diagnóstico inicial da base

In [ ]:
# Resumo por tabela: estrutura, duplicatas e valores ausentes
resumo_tabelas = []

for nome, df in sorted(dfs.items()):
    resumo_tabelas.append({
        "tabela": nome,
        "linhas": df.shape[0],
        "colunas": df.shape[1],
        "duplicatas": int(df.duplicated().sum()),
        "celulas_nulas": int(df.isna().sum().sum())
    })

resumo_tabelas = pd.DataFrame(resumo_tabelas)
display(resumo_tabelas)


In [ ]:
# Diagnóstico por coluna: tipo, ausentes e cardinalidade
diagnostico = []

for nome, df in sorted(dfs.items()):
    for coluna in df.columns:
        diagnostico.append({
            "tabela": nome,
            "coluna": coluna,
            "tipo": str(df[coluna].dtype),
            "nulos_qtd": int(df[coluna].isna().sum()),
            "nulos_pct": round(df[coluna].isna().mean() * 100, 2),
            "cardinalidade": int(df[coluna].nunique(dropna=True))
        })

diagnostico = pd.DataFrame(diagnostico)

display(
    diagnostico
    .sort_values(["nulos_pct", "cardinalidade"], ascending=[False, False])
    .head(30)
)


### Leitura inicial

A primeira análise mostra que:

- não foram encontradas linhas duplicadas nas tabelas;
- várias datas e timestamps são carregados inicialmente como texto (`object`);
- existem campos com valores ausentes que precisam ser interpretados antes de qualquer imputação;
- existem identificadores com alta cardinalidade, que devem ser usados para relacionamentos entre tabelas, mas não diretamente como variáveis categóricas em modelos;
- os principais cuidados estão na semântica dos dados, especialmente nos campos em que o `NaN` significa que determinada informação não se aplica.


## 2. Análise dos valores ausentes

In [ ]:
# Exemplo principal: latency_ms em ai_usage_logs
usage = dfs["ai_usage_logs.csv"].copy()

latency_por_status = (
    usage.groupby("request_status")["latency_ms"]
    .apply(lambda s: round(s.isna().mean() * 100, 2))
    .rename("latency_ms_nulo_pct")
)

display(latency_por_status)


### Decisão sobre `latency_ms`

A ausência de `latency_ms` não está distribuída igualmente entre os status das requisições. Ela aparece com maior frequência em situações de falha, como `error`, `timeout` e `rate_limited`.

Por isso, nesta etapa:

- **não preenchemos automaticamente a latência com média ou mediana**;
- a ausência será tratada como uma informação potencialmente relevante;
- futuramente poderá ser criada uma variável indicadora, como `latency_missing`.

Essa decisão evita apagar um possível sinal importante do comportamento da requisição.


In [ ]:
# Exemplos de possíveis NaNs estruturais

print("project_id ausente em ai_usage_logs:")
print(round(dfs["ai_usage_logs.csv"]["project_id"].isna().mean() * 100, 2), "%")

print("\nmodel_id ausente em ai_billing:")
print(round(dfs["ai_billing.csv"]["model_id"].isna().mean() * 100, 2), "%")

print("\nexpiration_date ausente em ai_licenses:")
print(round(dfs["ai_licenses.csv"]["expiration_date"].isna().mean() * 100, 2), "%")

print("\ncompletion_date ausente em training_records:")
print(round(dfs["training_records.csv"]["completion_date"].isna().mean() * 100, 2), "%")


### Exemplos de ausência estrutural

Nem todo valor ausente deve ser imputado.

- `project_id`: nem todo uso de IA está relacionado a um projeto;
- `model_id` em faturamento: cobranças por licença/assento podem não estar associadas a um modelo;
- `expiration_date`: alguns acordos podem não possuir data de expiração registrada;
- `completion_date` e `score`: não existem quando o treinamento não foi concluído.

Nesses casos, preencher automaticamente com média, mediana ou moda criaria uma informação que não existia originalmente.


## 3. Correção de tipos

In [ ]:
# Conversão de algumas colunas de data para o tipo correto
copias = {nome: df.copy() for nome, df in dfs.items()}

colunas_data = {
    "ai_usage_logs.csv": ["timestamp"],
    "employees.csv": ["hire_date"],
    "ai_licenses.csv": ["assigned_date", "expiration_date"],
    "training_records.csv": ["enrollment_date", "completion_date"],
    "security_events.csv": ["timestamp"],
    "projects.csv": ["start_date"],
    "pull_requests.csv": ["created_at", "merged_at"],
    "issues.csv": ["created_at", "closed_at"],
    "builds.csv": ["timestamp"],
    "deployments.csv": ["timestamp"],
    "approved_ai_tools.csv": ["approval_date"],
}

for tabela, colunas in colunas_data.items():
    for coluna in colunas:
        copias[tabela][coluna] = pd.to_datetime(
            copias[tabela][coluna],
            errors="coerce"
        )

print(copias["ai_usage_logs.csv"]["timestamp"].dtype)
print(copias["employees.csv"]["hire_date"].dtype)


A correção de tipos é feita antes de outras transformações, porque estatísticas e tratamentos só fazem sentido quando a coluna está representada corretamente.


## 4. Revisão da preparação financeira já realizada

In [ ]:
billing = dfs["ai_billing.csv"].copy()

# Taxas utilizadas na preparação anterior do grupo
fx = {
    "2025-06": 5.5465,
    "2025-07": 5.5279,
    "2025-08": 5.5465,
    "2025-09": 5.5279,
    "2025-10": 5.5465,
    "2025-11": 5.5279,
    "2025-12": 5.5465,
    "2026-01": 5.5279,
    "2026-02": 5.5465,
    "2026-03": 5.5279,
    "2026-04": 5.5465,
    "2026-05": 5.5279,
    "2026-06": 5.5465,
    "2026-07": 5.5279,
}

billing["currency_original"] = billing["currency"]
billing["total_cost_original"] = billing["total_cost"]

billing["fx_rate_to_brl"] = np.where(
    billing["currency"].eq("USD"),
    billing["billing_month"].map(fx),
    1.0
)

billing["total_cost_brl"] = (
    billing["total_cost"] * billing["fx_rate_to_brl"]
).round(2)

display(
    billing[
        [
            "billing_month",
            "total_cost_original",
            "currency_original",
            "fx_rate_to_brl",
            "total_cost_brl"
        ]
    ].head(10)
)


### Ajuste em relação à preparação anterior

A conversão USD → BRL realizada anteriormente pelo grupo foi mantida como parte da preparação.

A diferença é que, neste notebook, os valores originais são preservados em colunas separadas. Assim, a conversão pode ser auditada e refeita posteriormente sem perda do dado original.


## 5. Variáveis categóricas e identificadores

Para a modelagem futura, as variáveis serão separadas conforme sua natureza:

**Nominais — sem ordem**
- setor;
- departamento;
- cargo;
- ferramenta;
- aplicação.

Tratamento futuro: `OneHotEncoder`, de preferência dentro do Pipeline.

**Ordinais — possuem ordem**
- `seniority`;
- `company_size`;
- `severity`;
- `priority`.

Tratamento futuro: `OrdinalEncoder` com a ordem definida manualmente.

**Identificadores**
- `employee_id`;
- `company_id`;
- `department_id`;
- `usage_id`;
- `billing_id`;
- demais chaves.

Esses campos serão mantidos para relacionamentos e rastreabilidade, mas não devem ser submetidos diretamente a one-hot, pois possuem alta cardinalidade e podem fazer o modelo memorizar registros.


## 6. Registro das principais decisões

In [ ]:
decisoes = pd.DataFrame([
    {
        "campo": "latency_ms",
        "problema": "Valores ausentes",
        "decisao": "Não imputar automaticamente nesta etapa",
        "justificativa": "A ausência está relacionada a falhas e pode carregar informação."
    },
    {
        "campo": "project_id",
        "problema": "Alta quantidade de nulos",
        "decisao": "Tratar como ausência estrutural",
        "justificativa": "Nem todo uso de IA está associado a projeto."
    },
    {
        "campo": "model_id em ai_billing",
        "problema": "Valores nulos",
        "decisao": "Não imputar",
        "justificativa": "Cobranças por licença/assento podem não possuir modelo associado."
    },
    {
        "campo": "completion_date / score",
        "problema": "Valores nulos",
        "decisao": "Manter nulos quando treinamento não foi concluído",
        "justificativa": "A informação não existe nesses casos."
    },
    {
        "campo": "IDs",
        "problema": "Alta cardinalidade",
        "decisao": "Usar apenas para relacionamento/rastreabilidade",
        "justificativa": "Evita explosão da matriz e memorização."
    },
    {
        "campo": "Variáveis nominais",
        "problema": "Texto sem ordem",
        "decisao": "Aplicar OneHotEncoder futuramente",
        "justificativa": "Não existe ordem semântica entre as categorias."
    },
    {
        "campo": "Variáveis ordinais",
        "problema": "Categorias com ordem",
        "decisao": "Aplicar OrdinalEncoder futuramente",
        "justificativa": "A ordem deve ser declarada manualmente."
    },
    {
        "campo": "Tokens e requisições",
        "problema": "Possível forte assimetria",
        "decisao": "Avaliar log1p e escalonamento",
        "justificativa": "Evita que valores extremos dominem modelos sensíveis à escala."
    }
])

display(decisoes)


## 7. Preparação futura e prevenção de vazamento

As etapas que aprendem parâmetros dos dados — como imputação, escalonamento e codificação — **não serão aplicadas na base inteira antes da divisão entre treino e teste**.

Quando a modelagem for iniciada, essas transformações deverão ficar dentro de um `Pipeline`, ajustado apenas com o conjunto de treino.

Como a base possui histórico temporal, a avaliação futura também deve respeitar a ordem do tempo, evitando que registros futuros sejam usados para preparar ou prever registros passados.


## 8. Situação da variável-alvo

A base não possui um campo que indique diretamente se um colaborador, equipe ou departamento apresenta **dependência de IA**.

Por isso, nesta etapa, **não foi criada artificialmente uma variável-alvo**.

Antes da modelagem, ainda será necessário validar com o professor qual abordagem será utilizada, por exemplo:

- construir um indicador de risco com critérios de negócio previamente validados;
- utilizar técnicas não supervisionadas para identificar padrões e grupos atípicos;
- ou definir posteriormente uma variável-alvo com base em evidências adicionais.

Criar agora uma regra arbitrária como “quem usa mais IA é dependente” faria o modelo apenas reproduzir uma definição criada pelo próprio grupo.


## 9. Conclusão

Nesta etapa, a preparação foi concentrada em **entender a base antes de transformá-la**.

Foram identificados:

- tipos que precisam de correção;
- valores ausentes estruturais e informativos;
- variáveis categóricas nominais e ordinais;
- identificadores de alta cardinalidade;
- variáveis numéricas que poderão exigir transformação e escala;
- cuidados necessários para evitar vazamento de dados;
- necessidade de validação da definição de “dependência de IA” antes da construção final do dataset de modelagem.
